<a href="https://colab.research.google.com/github/raushankumar018/MLOps-Lab/blob/main/Task_4_Pipeline_with_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Import Libraries
import pandas as pd

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [5]:

# Cell 2: Upload Dataset
uploaded = files.upload()

Saving Titanic-Dataset.csv to Titanic-Dataset.csv


In [6]:
# Step 3 — Get Titanic Dataset
df = pd.read_csv("Titanic-Dataset.csv")

print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [7]:
# Step 4 — Understand the Dataset
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [8]:
# Step 5 — Select Features and Target
X = df[
    [
        "Pclass",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "Embarked"
    ]
]

y = df["Survived"]

In [9]:
# Step 6 — Define Numerical and Categorical Columns
numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

In [10]:
# Step 7 — Create Numerical Pipeline
numerical_pipeline = Pipeline(
    steps=[
        ("missing_values", SimpleImputer(strategy="median")),
        ("scaling", StandardScaler())
    ]
)

In [11]:
# Step 8 — Create Categorical Pipeline
categorical_pipeline = Pipeline(
    steps=[
        ("missing_values", SimpleImputer(strategy="most_frequent")),
        ("encoding", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [12]:
# Step 9 — Combine Both Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [13]:
# Step 10 — Add Logistic Regression
model_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)


In [14]:
# Step 11 — Split Training and Testing Data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [15]:
# Step 12 — Train the Complete Pipeline
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('missing_values',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare']),
                                                 ('categorical',
                                                  Pipeline(steps=[('missing_values',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoding',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [16]:
# Step 13 — Make Predictions
predictions = model_pipeline.predict(X_test)

print(predictions[:10])

[0 0 0 0 1 0 1 0 0 0]


In [17]:
# Step 14 — Calculate Accuracy
accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100)

Accuracy: 0.8044692737430168
Accuracy Percentage: 80.44692737430168


In [18]:
# Step 15 — Predict a New Passenger
new_passenger = pd.DataFrame(
    {
        "Pclass": [3],
        "Sex": ["male"],
        "Age": [25],
        "SibSp": [0],
        "Parch": [0],
        "Fare": [8.5],
        "Embarked": ["S"]
    }
)

In [19]:
# Predict:
prediction = model_pipeline.predict(new_passenger)

print("Prediction:", prediction[0])

Prediction: 0


In [20]:
# Step 16 — Save the Pipeline
import joblib

In [21]:
# Save:
joblib.dump(
    model_pipeline,
    "titanic_pipeline.pkl"
)

['titanic_pipeline.pkl']

In [22]:
# Step 17 — Load the Pipeline
loaded_pipeline = joblib.load(
    "titanic_pipeline.pkl"
)

In [23]:
# lets see
prediction = loaded_pipeline.predict(
    new_passenger
)

print("Prediction:", prediction[0])

Prediction: 0
